# Step 1.4 - EDA


## Section 1 - Setup


In [4]:
# Установка зависимостей (пропускается если уже установлены)
import importlib, subprocess, sys

def ensure_package(package, import_name=None):
    name = import_name or package
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install",
             package, "-q"])
        print(f"{package} installed!")

ensure_package("wordcloud")
ensure_package("loguru")
ensure_package("plotly")
ensure_package("pandas")
ensure_package("kaleido")

print("✅ All dependencies ready")


  Using cached wordcloud-1.9.6-cp312-cp312-win_amd64.whl.metadata (3.5 kB)
Using cached wordcloud-1.9.6-cp312-cp312-win_amd64.whl (307 kB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
# Dependencies are already checked in the previous cell


  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
  Using cached win32_setctime-1.2.0-py3-none-any.whl.metadata (2.4 kB)
Using cached loguru-0.7.3-py3-none-any.whl (61 kB)
Using cached win32_setctime-1.2.0-py3-none-any.whl (4.1 kB)
Note: you may need to restart the kernel to use updated packages.


In [7]:
import json
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, Image, Markdown, display
from wordcloud import WordCloud

ROOT = Path.cwd().resolve()
if not (ROOT / 'core').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.llm_client import GeminiLLMClient
from notebooks.export_eda import (
    build_conclusions_html,
    build_quality_heatmap,
    build_rows_per_source_figure,
    build_source_boxplot,
    build_source_pie_figure,
    build_source_stats_table,
    build_stopwords,
    build_text_length_histogram,
    build_top_words_figure,
    compute_quality_preview,
    compute_thematic_stats,
    load_inputs,
    save_wordcloud_image,
    source_distribution_frame,
)

df, domain_spec = load_inputs(ROOT)
client = GeminiLLMClient(config_path=str(ROOT / 'config.yaml'))
dataset_summary = client.build_dataset_summary(df)
stopwords = build_stopwords(domain_spec)
thematic_stats = compute_thematic_stats(df)

print(df.shape)
display(df.dtypes.to_frame('dtype'))


ImportError: cannot import name 'build_conclusions_markdown' from 'notebooks.export_eda' (D:\Projects\Data_Agent_Project\smart-data-pipeline\notebooks\export_eda.py)

## Section 2 - Dataset overview


In [ ]:
source_df = source_distribution_frame(df)
fig_source_bar = build_rows_per_source_figure(source_df, thematic_stats)
fig_source_bar.show()

fig_source_pie = build_source_pie_figure(source_df, thematic_stats)
fig_source_pie.show()


## Section 3 - Text lengths


In [ ]:
fig_length_hist = build_text_length_histogram(df)
fig_length_hist.show()

fig_length_box = build_source_boxplot(df)
fig_length_box.show()


## Section 4 - WordCloud


In [ ]:
reports_dir = ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

_ = save_wordcloud_image(
    df['text'].tolist(),
    stopwords,
    reports_dir / 'wordcloud_all.png',
    colormap='Blues',
)
domain_texts = df.loc[
    df['source'].apply(lambda name: name.startswith('rss_') or name in {'stackexchange_sailing', 'sailingforums'}),
    'text',
].tolist()
_ = save_wordcloud_image(
    domain_texts,
    stopwords,
    reports_dir / 'wordcloud_domain.png',
    colormap='viridis',
)

display(Image(filename=str(reports_dir / 'wordcloud_all.png')))
display(Image(filename=str(reports_dir / 'wordcloud_domain.png')))


## Section 5 - Data quality preview


In [ ]:
quality_df, table_df = compute_quality_preview(df)
fig_quality_heatmap = build_quality_heatmap(quality_df)
fig_quality_heatmap.show()

fig_source_table = build_source_stats_table(table_df)
fig_source_table.show()


## Section 6 - Top words by source


In [ ]:
fig_top_words = build_top_words_figure(df, stopwords)
fig_top_words.show()


## Section 7 - LLM hypotheses


In [2]:
hypotheses = client.generate_eda_hypotheses(dataset_summary)
(ROOT / 'reports' / 'eda_hypotheses.json').write_text(
    json.dumps(hypotheses, indent=2, ensure_ascii=False),
    encoding='utf-8',
)
display(Markdown('\n'.join(f'- {item}' for item in hypotheses)))


NameError: name 'client' is not defined

## Section 8 - Conclusions and recommendations


In [3]:
display(HTML(build_conclusions_html(thematic_stats, quality_df)))


NameError: name 'build_conclusions_markdown' is not defined